
# 01 — MIMIC-IV-ED Data Inventory and Quality Assessment

**Project:** Predicting Opioid Dispensing in Low Back Pain Emergency Visits  
**Phase:** 01 — Data Inventory & EDA  
**Dataset:** MIMIC-IV-ED v2.2  
**Unit of Analysis:** ED encounter (`stay_id`)

## Executive Summary

This notebook examines the structure, integrity, and quality of MIMIC-IV-ED data required for a retrospective machine learning study of documented opioid dispensing in ICD-defined low back pain encounters.

Using previously executed BigQuery SQL queries and locally exported CSV summaries, the analysis evaluates encounter-level relationships, triage feature availability, and medication-record timing.

The outputs identify data preparation requirements for subsequent cohort construction, opioid outcome definition, and predictive modeling.

Findings are based on database-wide exploratory analyses and should not be interpreted as characteristics of the final low back pain cohorts.

**Key Findings:** Encounter identifiers passed the examined structural integrity checks, and candidate triage variables showed relatively low SQL NULL missingness. However, pain score irregularities, extreme vital sign values, and out-of-window Pyxis timestamps require explicit handling in downstream data preparation.

**Conclusion:** These findings support proceeding to ICD-based cohort integration and outcome construction, subject to the identified data-quality limitations.

## Table of Contents

1. [Study Context & Data Sources](#1-study-context--data-sources)
2. [Data Structure & Encounter Integrity](#2-data-structure--encounter-integrity)
3. [Triage Feature Availability & Quality](#3-triage-feature-availability--quality)
4. [Temporal Integrity & Outcome Considerations](#4-temporal-integrity--outcome-considerations)
5. [Key Findings & Downstream Implications](#5-key-findings--downstream-implications)





In [21]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project paths
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results" / "01_eda"

# Reproducibility
RANDOM_SEED = 42

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

# Verify required local data directory
assert RESULTS_DIR.is_dir(), f"Missing results directory: {RESULTS_DIR}"

print("EDA results directory verified.")


EDA results directory verified.



## 1. Study Context & Data Sources

### 1.1 Research Context

This notebook supports the project *Predicting Opioid Dispensing in Low Back Pain Emergency Visits: Explainable Machine Learning and Clinical Phenotype Sensitivity*.

The study investigates whether routinely available emergency department (ED) triage information can predict documented opioid dispensing among encounters retrospectively identified using ICD-defined low back pain (LBP) phenotypes.

Two ICD-based cohort definitions will be evaluated:

- **C1 — Strict LBP:** The primary study cohort.
- **C2 — Expanded LBP:** A broader cohort used for sensitivity analysis.

The purpose of this initial exploratory data analysis (EDA) is to assess the structure and quality of the underlying data before cohort construction and predictive modeling.

### 1.2 Data Sources

**Dataset:** MIMIC-IV-ED v2.2 (Johnson et al., 2023)  
**Query platform:** Google BigQuery  
**Unit of analysis:** ED encounter (`stay_id`)

The analysis focuses on six MIMIC-IV-ED tables:

| Table | Role in the Study |
|---|---|
| `edstays` | ED encounter identifiers and visit information |
| `triage` | Structured triage features and chief complaint text |
| `diagnosis` | ICD-based LBP cohort identification |
| `vitalsign` | Repeated clinical observations and temporal data-quality checks |
| `pyxis` | Primary source for documented medication dispensing |
| `medrecon` | Context for medication-source coverage and documentation differences |

The final analytical dataset will contain one record per ED encounter. Tables containing multiple records per encounter must be appropriately aggregated before integration.

### 1.3 Scope and Objectives

This notebook evaluates four areas:

1. **Data structure and integrity:** Table granularity, identifier uniqueness, and referential integrity.
2. **Triage feature quality:** Missingness, distributions, and exploratory plausibility checks.
3. **Chief complaint availability:** Basic text completeness and characteristics relevant to later feature engineering.
4. **Temporal consistency:** ED encounter boundaries and medication-record timestamps relevant to outcome construction.

The analysis uses database-wide SQL summaries exported from BigQuery as CSV files. These results describe the available data infrastructure and do not represent the final C1 or C2 study cohorts.

This notebook does not establish clinical phenotype validity, define the final opioid outcome, or evaluate predictive model performance. Those tasks are addressed in subsequent project phases.

### 1.4 Analytical Workflow and Outputs

SQL queries are maintained separately under `sql/01_eda/`. Their aggregate outputs are exported to local CSV files under `results/01_eda/` and analyzed using Python.

The primary outputs of this notebook are:

- Verified encounter-level integration assumptions.
- A summary of candidate triage feature availability and data-quality concerns.
- Identification of temporal and documentation limitations relevant to opioid dispensing.
- Data preparation requirements for subsequent cohort integration, outcome construction, and machine learning.

No patient-level MIMIC data are intended for publication in the project repository.

### Reference

Johnson, A. E. W., et al. (2023). *MIMIC-IV-ED, version 2.2*. PhysioNet. https://physionet.org/content/mimic-iv-ed/2.2/



## 2. Data Structure & Encounter Integrity

### 2.1 Schema and Table Granularity

**Objective:** Examine the schemas and record granularity of six MIMIC-IV-ED tables to establish the data integration requirements for subsequent analysis.

**Method:** Review precomputed BigQuery summaries of column metadata, total record counts, and distinct ED encounters (`stay_id`).

**SQL sources:**
- `sql/01_eda/eda_01_schema.sql`
- `sql/01_eda/eda_02_table_granularity.sql`

**Key considerations:**
- `edstays` provides the encounter-level reference population.
- `triage` contains candidate predictors available from the triage record.
- `diagnosis` supports retrospective ICD-based cohort identification.
- `pyxis` provides medication dispensing records for subsequent outcome construction.
- `vitalsign` and `medrecon` provide additional data-quality and documentation context.

Tables with multiple records per encounter require aggregation or appropriate feature selection before integration to prevent unintended row multiplication.


In [22]:
# Load source schema and granularity summaries
schema_df = pd.read_csv(RESULTS_DIR / "eda_01_schema.csv")
granularity_df = pd.read_csv(
    RESULTS_DIR / "eda_02_table_granularity.csv"
)

# Summarize schema structure
schema_summary = (
    schema_df.groupby("table_name")
    .agg(
        column_count=("column_name", "count"),
        nullable_columns=("is_nullable", lambda x: (x == "YES").sum())
    )
    .reset_index()
)

display(schema_summary)

# Summarize encounter-level granularity
display(granularity_df)


,table_name,column_count,nullable_columns
0,_metadata,2,2
1,diagnosis,6,0
2,edstays,9,3
3,medrecon,9,7
4,pyxis,7,3
5,triage,11,9
6,vitalsign,11,9


,table_name,row_count,unique_stays,avg_rows_per_stay
0,diagnosis,899050,423989,2.12
1,edstays,425087,425087,1.00
2,medrecon,2987342,307196,9.72
3,pyxis,1586053,303361,5.23
4,triage,425087,425087,1.00
5,vitalsign,1564610,408146,3.83



#### Interpretation

The six source tables contain different levels of record granularity.

Both `edstays` and `triage` contain 425,087 records and 425,087 distinct ED encounters. In contrast, `diagnosis`, `vitalsign`, `pyxis`, and `medrecon` contain multiple records per documented encounter.

These results establish two requirements for downstream data integration:

1. The analytical dataset should preserve one record per `stay_id`.
2. Diagnosis and medication records must be aggregated to encounter-level indicators before joining them to the analytical dataset.

The presence of a record in `pyxis` does not establish opioid dispensing. Medication-specific outcome classification will be performed separately.


### 2.2 Primary Key Uniqueness & Referential Integrity

**Objective:** Validate encounter identifier uniqueness in the two encounter-level source tables and check whether records in related tables reference valid ED encounters.

**Method:** Two precomputed BigQuery queries assess:

1. **Key uniqueness:** Null and duplicate `stay_id` values in `edstays` and `triage`.
2. **Referential integrity:** Null `stay_id` values and records without a matching `edstays.stay_id` across five related tables.

**SQL sources:**
- `sql/01_eda/eda_03_key_validation.sql`
- `sql/01_eda/eda_04_referential_integrity.sql`

These checks assess structural data integrity rather than clinical data accuracy.


In [23]:
# Load precomputed key and referential integrity results
key_df = pd.read_csv(
    RESULTS_DIR / "eda_03_key_validation.csv"
)

ref_df = pd.read_csv(
    RESULTS_DIR / "eda_04_referential_integrity.csv"
)

# Display key validation results
display(key_df)

# Display referential integrity results
display(ref_df)


,table_name,total_rows,unique_stays,null_stay_ids,duplicate_rows
0,edstays,425087,425087,0,0
1,triage,425087,425087,0,0


,table_name,total_rows,unique_stays,null_stay_ids,unmatched_rows,unmatched_stays
0,diagnosis,899050,423989,0,0,0
1,medrecon,2987342,307196,0,0,0
2,pyxis,1586053,303361,0,0,0
3,triage,425087,425087,0,0,0
4,vitalsign,1564610,408146,0,0,0



#### Interpretation

The key validation confirmed that `edstays` and `triage` each contain 425,087 records with unique, non-null `stay_id` values.

Referential integrity checks identified no null or unmatched `stay_id` values across the five related tables (`triage`, `diagnosis`, `vitalsign`, `pyxis`, and `medrecon`).

These results support the use of `edstays` as the encounter-level reference table and `stay_id` as the primary integration key.

However, the absence of orphan records does not imply complete data coverage for every encounter. Related tables differ in encounter coverage, and one-to-many tables require aggregation before joining.

**Implication for downstream analysis:** Cohort construction should preserve one row per `stay_id`, with explicit uniqueness checks after encounter-level integration.


## 3. Triage Feature Availability & Quality

### 3.1 Triage Feature Missingness

**Objective:** Assess the availability of candidate triage-time predictors before feature preprocessing and model development.

**Method:** Examine field-level missingness across the full MIMIC-IV-ED `triage` table using a precomputed BigQuery summary. Missingness is defined as SQL `NULL`.

**SQL source:** `sql/01_eda/eda_05_triage_missingness.sql`

The assessment includes chief complaint, pain score, acuity, and six vital sign measurements.

This initial assessment does not evaluate whether non-NULL values are clinically plausible or suitable for modeling. These issues are examined in subsequent sections.


In [24]:

# Load triage field-level missingness summary
missingness_df = pd.read_csv(
    RESULTS_DIR / "eda_05_triage_missingness.csv"
)

# Prepare a concise summary
missingness_summary = (
    missingness_df
    .assign(
        variable=lambda df: df["variable_name"].str.removesuffix("_missing")
    )
    [["variable", "total_rows", "missing_count", "missing_pct"]]
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

display(missingness_summary)


,variable,total_rows,missing_count,missing_pct
0,temperature,425087,23415,5.51
1,o2sat,425087,20596,4.85
2,resprate,425087,20353,4.79
3,dbp,425087,19091,4.49
4,sbp,425087,18291,4.30
5,heartrate,425087,17090,4.02
6,pain,425087,12896,3.03
7,acuity,425087,6987,1.64
8,chiefcomplaint,425087,22,0.01



#### Interpretation

Across 425,087 ED encounters, all nine examined triage fields had SQL NULL missingness rates below 6%.

Chief complaint and acuity had particularly low NULL missingness (0.01% and 1.64%, respectively). Among the six vital sign fields, temperature had the highest missingness rate (5.51%).

These results indicate that the candidate triage predictors are generally well populated at the database level. However, non-NULL values may still contain nonstandard representations or implausible measurements.

**Implication for modeling:** Candidate triage fields can proceed to further quality assessment. Cleaning and missing-value handling will be defined within the training-data preprocessing pipeline rather than finalized from database-wide summaries alone.

These findings do not establish feature completeness within the final C1 or C2 cohorts.


### 3.2 Pain Score & Triage Acuity Quality

**Objective:** Assess the validity and distribution of two candidate triage predictors: pain score and acuity.

**Method:** Review precomputed BigQuery summaries using the following classification rules:

- **Pain score:** Numeric values between 0 and 10 (inclusive) are considered valid. Missing, non-numeric, and out-of-range values are identified separately.
- **Triage acuity:** Values from 1 to 5 are considered valid; NULL and out-of-range values are classified separately.

**SQL sources:**
- `sql/01_eda/eda_06_pain_distribution.sql`
- `sql/01_eda/eda_11_acuity_distribution.sql`

This assessment focuses on data representation and validity, not clinical interpretation of pain severity or associations with opioid dispensing.


In [25]:
# Load pain score and acuity summaries
pain_df = pd.read_csv(
    RESULTS_DIR / "eda_06_pain_distribution.csv"
)

acuity_df = pd.read_csv(
    RESULTS_DIR / "eda_11_acuity_distribution.csv"
)

# Aggregate pain score validity categories
pain_summary = (
    pain_df.groupby("pain_category", as_index=False)
    .agg(encounter_count=("record_count", "sum"))
)

pain_summary["percentage"] = (
    pain_summary["encounter_count"]
    / pain_summary["encounter_count"].sum()
    * 100
).round(2)

# Display concise summaries
print("Pain score validity:")
display(pain_summary)

print("Triage acuity distribution:")
display(acuity_df)


Pain score validity:


,pain_category,encounter_count,percentage
0,Missing,12896,3.03
1,Non-numeric,15218,3.58
2,Out of range,11204,2.64
3,Valid numeric,385769,90.75


Triage acuity distribution:


,acuity,acuity_status,encounter_count,encounter_pct
0,1.00,Valid,24019,5.65
1,2.00,Valid,139411,32.80
2,3.00,Valid,225066,52.95
3,4.00,Valid,28504,6.71
4,5.00,Valid,1100,0.26
5,NaN,Missing,6987,1.64



#### Interpretation

Pain score and triage acuity demonstrate different data-quality characteristics.

For pain score, 90.75% of encounters contain valid numeric values within the expected 0–10 range. The remaining 9.25% consist of missing (3.03%), non-numeric (3.58%), or out-of-range (2.64%) values. Therefore, SQL NULL missingness alone understates the preprocessing requirements for this variable.

Triage acuity is more consistently represented: 98.36% of encounters contain values within the expected 1–5 range, while 1.64% are missing. No out-of-range acuity values were observed.

**Implication for modeling:** Pain scores will require numeric conversion and explicit handling of nonstandard values. Acuity can be retained as a candidate predictor with appropriate missing-value handling. Final preprocessing choices will be implemented within the training pipeline to avoid data leakage.

These database-wide summaries do not establish the distributions or predictive value of either variable within the final C1 or C2 cohorts.



### 3.3 Vital Sign Distributions & Plausibility

**Objective:** Assess the distributions and potential data-quality issues of six structured triage vital sign variables considered for predictive modeling.

**Method:** Review precomputed BigQuery summaries covering:

1. Descriptive statistics, including approximate percentiles and extreme values.
2. Out-of-range values identified using exploratory plausibility thresholds.
3. Internal consistency of systolic and diastolic blood pressure measurements.

**SQL sources:**
- `sql/01_eda/eda_08_vitals_distribution.sql`
- `sql/01_eda/eda_09_vitals_plausibility.sql`
- `sql/01_eda/eda_10_blood_pressure_consistency.sql`

The plausibility thresholds serve as exploratory data-quality screening criteria rather than clinically validated diagnostic boundaries. Flagged observations are not automatically considered erroneous.

This assessment uses database-wide triage records and does not establish the distributions of the final C1 or C2 cohorts.


In [26]:

# Load precomputed vital sign quality summaries
vitals_dist = pd.read_csv(
    RESULTS_DIR / "eda_08_vitals_distribution.csv"
)

vitals_flags = pd.read_csv(
    RESULTS_DIR / "eda_09_vitals_plausibility.csv"
)

bp_quality = pd.read_csv(
    RESULTS_DIR / "eda_10_bp_consistency.csv"
)

# Combine descriptive statistics and plausibility flags
vitals_summary = vitals_dist.merge(
    vitals_flags[
        ["variable_name", "flagged_count", "flagged_pct_non_missing"]
    ],
    on="variable_name",
    validate="one_to_one"
)

# Select a concise set of quality indicators
vitals_summary = vitals_summary[
    [
        "variable_name",
        "median",
        "p01",
        "p99",
        "min_value",
        "max_value",
        "flagged_count",
        "flagged_pct_non_missing"
    ]
]

display(vitals_summary)

# Summarize blood pressure consistency
display(bp_quality)


,variable_name,median,p01,p99,min_value,max_value,flagged_count,flagged_pct_non_missing
0,dbp,77,44,115.00,0.00,661672,593,0.15
1,heartrate,84,51,134.00,1.00,1228,30,0.01
2,o2sat,99,92,100.00,0.00,9322,139,0.03
3,resprate,18,14,26.00,0.00,1820,47,0.01
4,sbp,133,90,198.00,1.00,151103,203,0.05
5,temperature,98,96,101.30,0.10,986,538,0.13


,total_encounters,complete_bp_pairs,sbp_below_dbp,sbp_equal_dbp,inconsistent_bp_pct
0,425087,405994,571,2,0.14



#### Interpretation

The six triage vital sign variables exhibit relatively concentrated central distributions but contain extreme observations that warrant preprocessing consideration.

Using the prespecified exploratory thresholds, fewer than 0.2% of non-missing observations were flagged for each vital sign. Additional blood pressure consistency checks identified 573 encounters (0.141% of complete BP pairs) with systolic blood pressure less than or equal to diastolic blood pressure.

These findings indicate that missingness alone does not fully characterize vital sign data quality. Extreme values may also distort descriptive statistics and affect subsequent predictive modeling.

**Implication for modeling:** The modeling pipeline should apply documented, reproducible rules for handling implausible values. Any values treated as invalid should be handled as missing rather than automatically corrected to assumed clinical values. Missing-value imputation and other learned preprocessing steps should be fitted using training data only.

The exploratory thresholds used here are not clinically validated exclusion criteria, and no encounters are excluded based on these findings. Final preprocessing will be applied consistently across C1 and C2.



### 3.4 Chief Complaint Data Availability & Basic Characteristics

**Objective:** Assess the availability and basic textual variability of ED chief complaint records before downstream feature engineering.

**Method:** Review a precomputed BigQuery frequency table of chief complaint expressions normalized using `LOWER(TRIM(chiefcomplaint))`. Summarize the number of distinct expressions, missing or blank entries, and the most frequently documented complaints.

**SQL source:**
- `sql/01_eda/eda_07_chiefcomplaint_frequency.sql`

This analysis describes database-wide chief complaint documentation. It does not establish an LBP phenotype or evaluate clinical classification accuracy. Detailed rule-based phenotyping and text feature engineering are addressed separately in the chief complaint notebook.


In [27]:

# Load normalized chief complaint frequencies
complaint_df = pd.read_csv(
    RESULTS_DIR / "eda_07_chiefcomplaint_frequency.csv"
)

# Database-wide summary
total_encounters = complaint_df["encounter_count"].sum()

non_empty = complaint_df["complaint_normalized"].notna()

complaint_summary = pd.DataFrame({
    "Metric": [
        "Total ED encounters",
        "Unique non-empty expressions",
        "NULL or blank complaints (CSV)",
        "NULL or blank percentage"
    ],
    "Value": [
        f"{total_encounters:,}",
        f"{complaint_df.loc[non_empty, 'complaint_normalized'].nunique():,}",
        f"{complaint_df.loc[~non_empty, 'encounter_count'].sum():,}",
        f"{complaint_df.loc[~non_empty, 'encounter_count'].sum() / total_encounters * 100:.3f}%"
    ]
})

display(complaint_summary)

# Most frequent documented expressions
top_complaints = (
    complaint_df.loc[non_empty]
    .nlargest(8, "encounter_count")
    [["complaint_normalized", "encounter_count", "encounter_pct"]]
    .reset_index(drop=True)
)

display(top_complaints)


,Metric,Value
0,Total ED encounters,"425,087"
1,Unique non-empty expressions,"58,361"
2,NULL or blank complaints (CSV),23
3,NULL or blank percentage,0.005%


,complaint_normalized,encounter_count,encounter_pct
0,abd pain,15850,3.73
1,chest pain,13911,3.27
2,s/p fall,7597,1.79
3,dyspnea,7422,1.75
4,headache,5125,1.21
5,wound eval,5089,1.20
6,si,4936,1.16
7,back pain,4876,1.15



#### Interpretation

Chief complaint documentation is highly complete at the database level. Among 425,087 ED encounters, 58,361 distinct non-empty expressions were observed after lowercase conversion and whitespace trimming.

The large number of distinct expressions indicates substantial textual variability despite the high field-level availability. Common expressions include abbreviated clinical terms, and the exact phrase `back pain` appeared in 4,876 encounters (1.15%).

These results support the need for consistent text preprocessing before using chief complaints as predictive features. However, frequency counts alone cannot establish whether an expression represents clinically confirmed low back pain.

**Implication for modeling:** Chief complaint text will be considered as an additional triage-time predictor in the ICD-defined C1 cohort. Rule-based indicators developed in the separate chief complaint phenotyping notebook may be reused for feature engineering, with text-derived model preprocessing fitted on training data only.

The present analysis does not use chief complaints to define the primary study cohort or evaluate phenotype classification performance.



## 4. Temporal Integrity & Outcome Considerations

### 4.1 ED Encounter Duration & Timestamp Quality

**Objective:** Evaluate the completeness and chronological consistency of ED encounter timestamps before using encounter boundaries to assess medication-related records.

**Method:** Review a precomputed BigQuery summary of `edstays.intime`, `edstays.outtime`, and their time differences. The analysis examines missing timestamps, nonpositive durations, unusually long encounters, and descriptive duration statistics.

**SQL source:**
- `sql/01_eda/eda_12_ed_stay_duration.sql`

ED arrival and departure timestamps provide encounter-level reference boundaries. They do not establish the precise completion time of triage or the actual time of medication administration.

This is a database-wide quality assessment; no final C1/C2 eligibility or exclusion decisions are made here.


In [28]:

# Load ED encounter duration summary
duration_df = pd.read_csv(
    RESULTS_DIR / "eda_12_ed_stay_duration.csv"
)

# SQL 12 produces a single database-wide summary row
assert len(duration_df) == 1

d = duration_df.iloc[0]
n = int(d["total_encounters"])

# Timestamp quality checks
quality_summary = pd.DataFrame({
    "Metric": [
        "Total ED encounters",
        "Missing arrival timestamps",
        "Missing departure timestamps",
        "Negative duration",
        "Zero duration",
        "Duration > 24 hours",
        "Duration > 48 hours",
        "Duration > 72 hours"
    ],
    "Count": [
        n,
        d["missing_intime"],
        d["missing_outtime"],
        d["negative_duration"],
        d["zero_duration"],
        d["over_24h"],
        d["over_48h"],
        d["over_72h"]
    ]
})

quality_summary["Percentage"] = (
    quality_summary["Count"] / n * 100
).round(3)

display(quality_summary)

# Descriptive duration statistics
duration_summary = pd.DataFrame({
    "Statistic": [
        "Mean",
        "Median",
        "25th percentile",
        "75th percentile",
        "95th percentile",
        "99th percentile",
        "Minimum",
        "Maximum"
    ],
    "Hours": [
        d["mean_hours"],
        d["median_hours"],
        d["p25_hours"],
        d["p75_hours"],
        d["p95_hours"],
        d["p99_hours"],
        d["min_hours"],
        d["max_hours"]
    ]
})

display(duration_summary)


,Metric,Count,Percentage
0,Total ED encounters,"425,087.00",100.00
1,Missing arrival timestamps,0.00,0.00
2,Missing departure timestamps,0.00,0.00
3,Negative duration,6.00,0.00
4,Zero duration,53.00,0.01
5,Duration > 24 hours,"11,896.00",2.80
6,Duration > 48 hours,"1,249.00",0.29
7,Duration > 72 hours,301.00,0.07


,Statistic,Hours
0,Mean,7.16
1,Median,5.47
2,25th percentile,3.53
3,75th percentile,8.32
4,95th percentile,19.88
5,99th percentile,30.53
6,Minimum,-22.73
7,Maximum,493.07



#### Interpretation

ED arrival and departure timestamps are complete across all 425,087 encounters. The median ED duration is 5.47 hours, with a mean of 7.16 hours.

A small number of encounters have nonpositive durations: six negative-duration encounters and 53 zero-duration encounters. In addition, 2.80% of encounters exceed 24 hours.

These findings indicate that ED encounter boundaries are generally available but require chronological validity checks before being used to evaluate medication-event timing.

**Implication for downstream analysis:** P4/P5 should explicitly handle invalid ED intervals when constructing encounter-level medication outcomes. Arrival and departure timestamps may define an operational observation window, but their availability does not establish the exact time of triage completion or medication administration.

No cohort exclusions or opioid outcome labels are determined in this exploratory assessment.



### 4.2 Event Timestamp Consistency

**Objective:** Assess whether recorded event timestamps in `vitalsign`, `pyxis`, and `medrecon` fall within the corresponding ED encounter boundaries.

**Method:** Compare each event's `charttime` with `edstays.intime` and `edstays.outtime`. Events are classified into five mutually exclusive categories:

- **Within ED stay:** Timestamp falls within the recorded arrival–departure interval, including boundaries.
- **Before arrival:** Timestamp precedes ED arrival.
- **After departure:** Timestamp follows ED departure.
- **Missing timestamp:** Event timestamp is NULL.
- **Invalid ED interval:** ED arrival/departure timestamps are missing or departure precedes arrival.

**SQL source:** `sql/01_eda/eda_13_event_timestamp_validation.sql`

Percentages are calculated at the event-record level, not the encounter level. An encounter may contain events in multiple categories.

This assessment identifies temporal inconsistencies in recorded data but does not establish when clinical actions actually occurred.


In [29]:

# Load event timestamp classification results
timing_df = pd.read_csv(
    RESULTS_DIR / "eda_13_event_timestamp_validation.csv"
)

# Summarize event counts and percentages
timing_summary = timing_df.pivot(
    index="table_name",
    columns="timestamp_status",
    values="event_pct"
)

# Reorder categories for readability
status_order = [
    "Within ED stay",
    "Before arrival",
    "After departure",
    "Missing timestamp",
    "Invalid ED interval"
]

timing_summary = timing_summary.reindex(
    columns=status_order,
    fill_value=0
)

timing_summary["Total events"] = (
    timing_df.groupby("table_name")["event_count"].sum()
)

timing_summary = timing_summary[
    ["Total events"] + status_order
].reset_index()

display(timing_summary.round(3))


timestamp_status,table_name,Total events,Within ED stay,Before arrival,After departure,Missing timestamp,Invalid ED interval
0,medrecon,2987342,98.87,0.00,1.13,0,0.00
1,pyxis,1586053,90.88,0.05,9.07,0,0.00
2,vitalsign,1564610,91.44,0.07,8.50,0,0.00



#### Interpretation

Most recorded events fall within their corresponding ED encounter intervals. However, temporal discrepancies are present across all three event tables.

Among 1,586,053 Pyxis event records, 143,854 (9.07%) have timestamps after recorded ED departure, affecting 26,702 distinct encounters. Post-departure records are also observed in `vitalsign` (8.50% of events) and `medrecon` (1.13%).

These findings demonstrate that event timestamps do not always align with encounter boundaries. The available summaries do not establish whether such discrepancies reflect delayed documentation, operational workflows, or other data-generation processes.

**Implication for outcome construction:** The P5 opioid dispensing pipeline must prespecify an observation window and explicitly determine how to handle Pyxis events outside the recorded ED interval. The effect on opioid-positive labels cannot be quantified until qualifying opioid medications have been identified.

Pyxis records represent documented dispensing-related events and should not be interpreted as confirmed medication administration or prescribing.

No events are excluded or reassigned in this exploratory analysis.



### 4.3 Post-Departure Event Timing

**Objective:** Characterize the time offsets of Pyxis and vital sign records documented after ED departure to inform downstream medication outcome construction.

**Method:** Review precomputed BigQuery summaries of positive time differences between event `charttime` and ED `outtime`. Post-departure events are grouped into five intervals: ≤15 minutes, 15–60 minutes, 1–6 hours, 6–24 hours, and >24 hours.

**SQL source:**
- `sql/01_eda/eda_14_post_departure_offsets.sql`

Percentages are calculated separately for each source table using post-departure event records as the denominator. Unique encounter counts within each interval are not additive because an encounter may contain events in multiple intervals.

These time offsets describe recorded timestamps, not verified delays in clinical documentation or medication administration.


In [30]:

# Load post-departure timing summary
offset_df = pd.read_csv(
    RESULTS_DIR / "eda_14_post_departure_offsets.csv"
)

# Validate event totals against Section 4.2
expected_totals = {
    "pyxis": 143_854,
    "vitalsign": 132_926
}

actual_totals = (
    offset_df.groupby("table_name")["event_count"].sum()
    .to_dict()
)

assert actual_totals == expected_totals, (
    f"Unexpected post-departure totals: {actual_totals}"
)

# Present event-level distribution
offset_summary = offset_df[
    [
        "table_name",
        "delay_category",
        "event_count",
        "pct_of_post_departure_events"
    ]
].copy()

offset_summary.columns = [
    "Source",
    "Time after departure",
    "Event count",
    "Percentage (%)"
]

display(offset_summary)

# Summarize events occurring 1–24 hours after departure
middle_intervals = ["03: 1-6 hours", "04: 6-24 hours"]

pyxis_later_pct = offset_df.loc[
    (offset_df["table_name"] == "pyxis")
    & (offset_df["delay_category"].isin(middle_intervals)),
    "pct_of_post_departure_events"
].sum()

print(
    f"Pyxis events 1–24 hours after departure: "
    f"{pyxis_later_pct:.2f}%"
)


,Source,Time after departure,Event count,Percentage (%)
0,pyxis,01: <=15 min,10319,7.17
1,pyxis,02: 15-60 min,18991,13.20
2,pyxis,03: 1-6 hours,44547,30.97
3,pyxis,04: 6-24 hours,53497,37.19
4,pyxis,05: >24 hours,16500,11.47
5,vitalsign,01: <=15 min,15761,11.86
6,vitalsign,02: 15-60 min,20940,15.75
7,vitalsign,03: 1-6 hours,45855,34.50
8,vitalsign,04: 6-24 hours,40992,30.84
9,vitalsign,05: >24 hours,9378,7.06


Pyxis events 1–24 hours after departure: 68.16%



#### Interpretation

Post-departure records occur across a range of time offsets in both Pyxis and vital sign data.

Among 143,854 post-departure Pyxis events, 68.16% were recorded between 1 and 24 hours after ED departure, while 11.47% occurred more than 24 hours afterward. Similar temporal discrepancies were observed in vital sign records.

These findings indicate that post-departure timestamps cannot be explained solely by a brief recording delay. However, the available data do not establish the underlying clinical or documentation workflows.

**Implication for outcome construction:** P5 must define a reproducible ED observation window before generating opioid dispensing labels. Post-departure Pyxis events require explicit handling, but their effect on opioid-positive encounter counts remains unknown until medication mapping is completed.

No additional temporal sensitivity experiments are conducted in P1.



## 5. Key Findings & Downstream Implications

### 5.1 Summary of Key Findings

The database-wide exploratory analysis assessed the structural integrity, triage feature quality, and temporal consistency of MIMIC-IV-ED v2.2. The main findings are summarized below.

| Domain | Key Finding | Research Implication |
|---|---|---|
| Encounter structure | 425,087 ED encounters; unique, non-null `stay_id` values in `edstays` and `triage` | Supports encounter-level integration |
| Referential integrity | No orphan records identified in the five related tables | Supports joining through `stay_id`, with aggregation where required |
| Triage availability | All nine examined triage fields have SQL NULL missingness below 6% | Candidate features are generally available |
| Pain score quality | 9.25% of values are missing, non-numeric, or outside 0–10 | Requires explicit numeric conversion and invalid-value handling |
| Vital sign quality | Exploratory plausibility checks identified extreme values and some inconsistent BP pairs | Requires documented preprocessing rules |
| Chief complaint | 58,361 distinct non-empty expressions after basic text normalization | Supports further text feature engineering in P3 |
| ED timestamps | Complete arrival/departure timestamps; 59 nonpositive-duration encounters | Requires encounter interval validation |
| Pyxis timing | 9.07% of Pyxis event records occur after recorded ED departure | Requires a prespecified dispensing observation window |

These results demonstrate that the source tables provide a workable foundation for encounter-level analysis while identifying data-quality issues that must be addressed before constructing the final modeling dataset.

### 5.2 Implications for Downstream Analysis

The findings inform three subsequent stages of the research workflow.

**P4 — Encounter-Level Cohort Integration**

- Construct ICD-defined C1 and C2 cohorts using the versioned phenotype codebook.
- Preserve one record per `stay_id` after joining encounter-level sources.
- Confirm adult eligibility and patient identifiers.
- Assess triage feature availability within the final cohorts.

**P5 — Opioid Dispensing Outcome Construction**

- Develop and review a reproducible opioid medication mapping.
- Define the operational ED observation window before assigning labels.
- Handle invalid encounter intervals and out-of-window Pyxis records explicitly.
- Interpret the outcome as documented opioid dispensing, not confirmed administration or prescribing.

**P6 — Predictive Modeling**

- Restrict predictors to information available at the intended triage prediction point.
- Apply consistent pain score and vital sign cleaning rules.
- Fit imputation, encoding, and text transformations using training data only.
- Use patient-level data partitioning to prevent patients from appearing in both training and test sets.

The P1 findings identify methodological requirements but do not determine final cohort eligibility, opioid-positive labels, or model performance.

### 5.3 Limitations & Conclusion

This analysis is based on database-wide aggregate summaries rather than the final adult ICD-defined low back pain cohorts. Data availability and quality may differ within C1 and C2.

Exploratory plausibility thresholds identify potentially problematic measurements but do not provide independent clinical validation. Similarly, timestamp discrepancies cannot establish the underlying documentation or medication workflow.

**Conclusion:** MIMIC-IV-ED provides identifiable ED encounters, generally well-populated triage fields, and medication dispensing records suitable for further investigation. The principal preparation requirements are encounter-level aggregation, explicit handling of nonstandard predictor values, and a reproducible medication observation window.

These findings support proceeding to cohort integration and outcome construction without expanding the exploratory analysis beyond its intended scope.
